In [1]:
from datasets import load_dataset, Dataset
from operator import truediv
from datasets import concatenate_datasets
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, EvalPrediction
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, hamming_loss
import torch
import numpy as np
import pandas as pd
from accelerate import Accelerator
exp_name = 'baseline_od'
device = Accelerator.device
accelerator = Accelerator()
base_model = "roberta-base"
EXP = 'cost' #should be perf or cost
tokenizer = AutoTokenizer.from_pretrained(base_model)
ds = load_dataset("CARROT-LLM-Routing/SPROUT-o3mini")
## CORRECTLY FORMAT DATA

ds_train = ds['train']
ds_test = ds['test']
ds_validation = ds['validation']
all_models = ds['train'].to_pandas().columns.tolist()[6:]
print(all_models)
train_inputs = ds_train['prompt']
val_inputs = ds_validation['prompt']
test_inputs = ds_test['prompt']
def get_labels(split, models, mode):
    labels = []
    if mode == 'cost':
        for m in models:
            tmp = [sample['num_output_tokens'] for sample in split[m]]
            labels.append(tmp)
        return np.array(labels).T
    else:
        for m in models:
            tmp = [sample['score'] for sample in split[m]]
            labels.append(tmp)
        return np.array(labels).T
if EXP == 'cost':
    train_labels = get_labels(ds_train, all_models, 'cost')
    val_labels = get_labels(ds_validation, all_models, 'cost')
    test_labels = get_labels(ds_test, all_models, 'cost')
    model_max_token_counts = [1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000]
    data_train = Dataset.from_dict({"input": train_inputs, "count": train_labels})
    data_val = Dataset.from_dict({"input": val_inputs, "count": val_labels})
    data_test = Dataset.from_dict({"input": test_inputs, "count": test_labels})
    def preprocess_function(examples):
        floating_counts = []
        for count_by_case in examples["count"]:
            floating_counts.append(list(map(truediv, count_by_case, model_max_token_counts)))
        examples = tokenizer(examples["input"], truncation=True)
        examples["label"] = floating_counts
        return examples
    tokenized_train = data_train.map(preprocess_function, batched=True, remove_columns=["count"])
    tokenized_val = data_val.map(preprocess_function, batched=True, remove_columns=["count"])
    tokenized_test = data_test.map(preprocess_function, batched=True, remove_columns=["count"])
    tokenized_train.set_format("torch")
    tokenized_val.set_format("torch")
    tokenized_test.set_format("torch")
else:
    train_labels = get_labels(ds_train, all_models, 'perf')
    val_labels = get_labels(ds_validation, all_models, 'perf')
    test_labels = get_labels(ds_test, all_models, 'perf')
    data_train = Dataset.from_dict({"input": train_inputs, "label": train_labels})
    data_val = Dataset.from_dict({"input": val_inputs, "label": val_labels})
    data_test = Dataset.from_dict({"input": test_inputs, "label": test_labels})
    def preprocess_function(examples):
        return tokenizer(examples["input"], truncation=True)
    tokenized_train = data_train.map(preprocess_function, batched=True)
    tokenized_val = data_val.map(preprocess_function, batched=True)
    tokenized_test = data_test.map(preprocess_function, batched=True)
    tokenized_train.set_format("torch")
    tokenized_val.set_format("torch")
    tokenized_test.set_format("torch")

## TRAIN COST/PERF PREDICTION
def compute_metrics_for_regression(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    mse = np.mean(np.square(labels-logits))
    return {"mse": mse}
def multi_label_metrics(predictions, labels, threshold=0.5):
    # first, apply sigmoid on predictions which are of shape (batch_size, num_labels)
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))
    # next, use threshold to turn them into integer predictions
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= threshold)] = 1
    # finally, compute metrics
    y_true = np.zeros(labels.shape)
    y_true[np.where(labels >= threshold)] = 1 
    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro')
    roc_auc = roc_auc_score(y_true, y_pred, average = 'micro')
    accuracy = accuracy_score(y_true, y_pred)
    # return as dictionary
    metrics = {'f1': f1_micro_average,
               'roc_auc': roc_auc,
               'accuracy': accuracy,
               'all_accuracy': (y_true == y_pred).mean()}
    return metrics


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
if EXP == 'cost':
    model = AutoModelForSequenceClassification.from_pretrained(base_model, 
                                                           problem_type="regression", 
                                                           num_labels=len(all_models),
                                                           )
    batch_size = 16
    metric_name = 'mse'
    num_train_epochs = 5
    args = TrainingArguments(
        exp_name + "-router-" + base_model,
        evaluation_strategy = "epoch",
        save_strategy = "epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model=metric_name,
        greater_is_better=False,
        save_total_limit=1,
        logging_dir='./fmselect-logs',
        logging_steps=10,
        report_to='none'
    )
    trainer = Trainer(
        model,
        args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics_for_regression,
        data_collator=data_collator
    )
    trainer.train()

else:
    model = AutoModelForSequenceClassification.from_pretrained(base_model, 
                                                           problem_type="multi_label_classification", 
                                                           num_labels=len(all_models), device_map = 'auto') #reference_compile=False)
    batch_size = 16
    metric_name = 'all_accuracy'
    num_train_epochs = 5
    args = TrainingArguments(
        exp_name + "-router-" + base_model,
        evaluation_strategy = "epoch",
        save_strategy = "epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model=metric_name,
        greater_is_better=True,
        save_total_limit=1,
        logging_dir='./fmselect-logs',
        logging_steps=10,
        report_to = 'none')
    def compute_metrics(p: EvalPrediction):
        preds = p.predictions[0] if isinstance(p.predictions, 
                tuple) else p.predictions
        result = multi_label_metrics(
            predictions=preds, 
            labels=p.label_ids)
        return result
    trainer = Trainer(
        model,
        args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        data_collator=data_collator)
    trainer.train()

['openai-o3-mini', 'wxai-llama-3-405b-instruct', 'wxai-llama-3-3-70b-instruct', 'wxai-llama-3-2-3b-instruct', 'wxai-llama-3-2-1b-instruct', 'wxai-llama-3-1-8b-instruct', 'wxai-llama-3-1-70b-instruct', 'aws-titan-text-premier-v1', 'wxai-mixtral-8x7b-instruct-v01', 'aws-claude-3-5-sonnet-v1', 'openai-gpt-4o-mini', 'openai-gpt-4o', 'wxai-granite-3-8b-instruct-8k-max-tokens', 'wxai-granite-3-2b-instruct-8k-max-tokens']


Map:   0%|          | 0/30301 [00:00<?, ? examples/s]

Map:   0%|          | 0/6493 [00:00<?, ? examples/s]

Map:   0%|          | 0/6494 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/skunk/routing2/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_140129/722317140.py:128: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/home/skunk/routing2/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return

Epoch,Training Loss,Validation Loss,Mse
1,1.051300,0.966218,0.966313
2,1.231800,0.965187,0.965257
3,0.825000,0.946590,0.946680
4,1.019100,0.955795,0.955899
5,0.987700,0.960254,0.960357


/home/skunk/routing2/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/skunk/routing2/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/skunk/routing2/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/skunk/routing2/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/skunk/routing2/lib/python3.10/site-packages/torch/nn/p

In [2]:
if EXP == 'perf':
    trainer.save_model("./HFPERFo3")
if EXP == 'cost':
    trainer.save_model("./HFcosto3")